In [ ]:
# CNN(Convolutional NN)
#   kernel/filter로 긁어가면서 
#       -> 주변만 연결 
#       -> 신경망을 적절히 잘 끊어가면서 위치 정보를 살림
#   다양한 특성을 가진 kernel여러개로 긁어서 channel축으로 쌓아
#       -> 다양한 특성 추출 가능
#   padding : 데이터 바깥쪽에 추가해서, kernel로 긁어도 사이즈 줄어들지 않게
#   stride : kernel 얼마씩 움직일지
#   pooling(max pooling/avg pooling) : 중요한 특징만 골라내서
#   여러층의 CNN을 통과하고 마지막에 FC Layer(Fully Connected Layer)통과
#       -> 적절히 끊어가면서 위치별로 봤던거, 최종적으로는 다 보고 결론내게

In [1]:
import torch
from torch import nn, optim, accelerator
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
from random import random

In [2]:
if accelerator.is_available():
    device = accelerator.current_accelerator()
else:
    device = "cpu"

print(device)

cpu


In [3]:
# 전체 데이터 30%정도는 학습용x, 테스트용
def getImg(folder, w, h, yData):
    trainData = []
    trainLabel = []
    testData = []
    testLabel = []
    for i, f in enumerate(os.listdir(folder)):
        data = cv2.imread(folder + f, cv2.IMREAD_COLOR)
        data = cv2.resize(data, (w, h))
        if random() < 0.3:
            testData.append(np.array(data, dtype=np.float32))
            testLabel.append(yData[i])
        else:
            trainData.append(np.array(data, dtype=np.float32))
            trainLabel.append(yData[i])
    return trainData, trainLabel, testData, testLabel

In [4]:
class KwonBunsikDataset(Dataset):
    def __init__(self, data, label, numClasses):
        super().__init__()
        self.datas = torch.from_numpy(np.array(data)) # 18개 x 50행 x 100열 x 3색
        self.datas = self.datas.permute(0, 3, 1, 2) # 18 x 50 x 100 x 3 -> 18 x 3 x 50 x 100
        self.labels = torch.from_numpy(np.array(label))
        self.labels = torch.nn.functional.one_hot(self.labels, numClasses)
        self.labels = self.labels.type(torch.float32)

    def __len__(self):
        return len(self.datas) # 전체 갯수 리턴

    def __getitem__(self, index):
        return self.datas[index], self.labels[index] # (데이터, 라벨) 튜플형태로 리턴

In [6]:
%pwd

'/mnt/batch/tasks/shared/LS_root/mounts/clusters/teacher023/code/Users/teacher02/Feb02_1_DeepLearning'

In [13]:
label = ["떡볶이", "오뎅", "김밥", "튀김", "순대"]
yData = [0,0,0,0,0, 1,1,1,1,1, 2,2,2,2,2, 3,3,3,3,3, 4,4,4,4,4]
trainData, trainLabel, testData, testLabel = getImg("./Users/teacher02/Feb02_1_DeepLearning/bunsikMenu/", 100, 50, yData)

trainDataset = KwonBunsikDataset(trainData, trainLabel, 5)
testDataset = KwonBunsikDataset(testData, testLabel, 5)
trainDataLoader = DataLoader(trainDataset, 3, True) # trainDataset에서 3개씩(batch size) 순서섞어서
testDataLoader = DataLoader(testDataset, 3, True)

In [14]:
class KwonBunsikCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.kbcnn = nn.Sequential(
            nn.Conv2d(3, 1000, 5), # in채널(R, G, B), out채널, 5
            nn.ReLU(),
            nn.Conv2d(1000, 500, 5),
            nn.ReLU(),
            nn.Conv2d(500, 100, 5),
            nn.ReLU(),
            nn.Conv2d(100, 50, 5),
            nn.ReLU()
        )
        self.f = nn.Flatten()
        self.nn = nn.Sequential(
            nn.Linear(142800, 10),
            nn.ReLU(),
            nn.Linear(10, 5)
        )

    def forward(self, x):
        myModel = self.kbcnn(x)
        myModel = self.f(myModel)
        myModel = self.nn(myModel)
        return myModel

In [15]:
model = KwonBunsikCNN().to(device)

lossFn = nn.CrossEntropyLoss()
o = optim.Adam(model.parameters(), lr=0.001)

In [16]:
model.train()

for epoch in range(3):
    for x, y in trainDataLoader:
        x = x.to(device)
        y = y.to(device)

        predY = model(x)
        l = lossFn(predY, y)

        o.zero_grad()
        l.backward()
        o.step()

    print(l.item())

1.5930649042129517
1.6675444841384888
1.7279716730117798


In [ ]:
# 정확도 테스트
# 전체 데이터 25개
#   70%정도는 학습용으로 사용, 30%정도는 테스트용으로 빼둠
# 그 테스트용 데이터가 떡볶이인데, AI가 떡볶이라고 예측
model.eval()
size = len(testDataLoader.dataset) # 테스트용 데이터 전체 개수
ok = 0  # 제대로 예측한 개수
with torch.no_grad(): # GD할 필요 없으니
    for x, y in testDataLoader:
        x = x.to(device)
        y = y.to(device)

        predY = model(x)
        ok += (predY.argmax(1) == y.argmax(1)).type(torch.float32).sum().item()
print(ok / size)

0.75


In [ ]:
# 예측
f = cv2.imread("./Users/teacher02/Feb02_1_DeepLearning/test.png", cv2.IMREAD_COLOR)
f = cv2.resize(f, (100, 50))
f = np.array([f], dtype=np.float32)
predData = torch.from_numpy(f).permute(0,3,1,2)
result = model(predData)
result = nn.Softmax(dim=1)(result)
result = result.argmax().item()
print(label[result])

오뎅
